<a href="https://colab.research.google.com/github/Chemistbp/report-generator-from-BI/blob/main/Report_maker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install python-pptx

In [ ]:
from datetime import datetime
import os
from pptx import Presentation
from pptx.enum.text import PP_ALIGN
from pptx.util import Inches
from pptx.enum.shapes import MSO_SHAPE_TYPE


In [ ]:
# Открываем
input_pptx = "/content/fixed_presentation.pptx"
prs = Presentation(input_pptx)



In [ ]:
# Коррекция титульного слайда
slide_1 = prs.slides[0]
# Изменим заголовок в шаблоне на актуальный месяц
months_ru = {
    1: "январь", 2: "февраль", 3: "март", 4: "апрель", 5: "май", 6: "июнь",
    7: "июль", 8: "август", 9: "сентябрь", 10: "октябрь", 11: "ноябрь", 12: "декабрь"
}

# Форматирование текущей даты
now = datetime.now()
if now.day <=5:
    actual_month = now.month - 1
else:
    actual_month = now.month

# Собираем строку вручную
formatted_date = f"{months_ru[actual_month]} {now.year}".upper()
print(formatted_date)
if slide_1.shapes.title:
    slide_1.shapes.title.text = f"Отчет за месяц \n {formatted_date}"
    for paragraph in slide_1.shapes.title.text_frame.paragraphs:
        paragraph.alignment = PP_ALIGN.CENTER



СЕНТЯБРЬ 2026


In [ ]:
def picture_replace(slide, new_image_path):
# Вставляем новые картинки

    for shape in slide.shapes:
        # Проверяем, является ли объект картинкой
        if shape.shape_type == MSO_SHAPE_TYPE.PICTURE:
            print("да")
            # 1. Получаем внутренний ID картинки на слайде
            slide_part = shape.part
            rId = shape._element.blip_rId
            image_part = slide_part.related_part(rId)

            # 2. Читаем файл новой картинки в байтах
            with open(new_image_path, "rb") as f:
                new_image_bytes = f.read()

            # 3. Перезаписываем бинарные данные напрямую в хранилище презентации
            image_part.blob = new_image_bytes

            #break  # Уберите break, если нужно заменить ВСЕ картинки на слайде

In [ ]:
def picture_replace_by_name(slide, target_shape_name, new_image_path):
    for shape in slide.shapes:
        # Проверяем тип и сверяем имя объекта
        if shape.shape_type == MSO_SHAPE_TYPE.PICTURE and shape.name == target_shape_name:
            print(f"Заменяем картинку с именем: {shape.name}")

            # Читаем новые байты
            with open(new_image_path, "rb") as f:
                new_image_bytes = f.read()

            # Прямая перезапись blob
            shape.part.related_part(shape._element.blip_rId).blob = new_image_bytes
            return  # Нашли и заменили, можно выходить

In [ ]:
#Функция смены картинки в слайде
# Будем собирать картинки для удаления, чтобы не ломать цикл при обходе shapes
def picture_replace_new(slide, new_image_path):

    images_to_replace = []

    for shape in slide.shapes:
        if shape.shape_type == MSO_SHAPE_TYPE.PICTURE:
            images_to_replace.append(shape)

    # Заменяем найденные картинки
    for old_shape in images_to_replace:
        # 1. Запоминаем точные координаты и размеры старой картинки
        left = old_shape.left
        top = old_shape.top
        width = old_shape.width
        height = old_shape.height

        # 2. Удаляем старую картинку из структуры слайда
        sp = old_shape._element
        sp.getparent().remove(sp)

        # 3. Вставляем новую картинку на те же координаты
        slide.shapes.add_picture(new_image_path, left, top, width, height)
        print("Картинка успешно заменена через перезапись объекта.")
        break  # Уберите, если нужно заменить ВСЕ картинки на слайде

Эта часть кода заменяет картнки в презентации

In [ ]:
slide = prs.slides[4]
new_image_path = "/content/IQDATA MarketData_CNT-N, RUR, month_new.png"
picture_replace(slide, new_image_path)
slide = prs.slides[5]
new_image_path = "/content/IQDATA MarketData_NWR-N, RUR, month_new.png"
picture_replace(slide, new_image_path)
slide = prs.slides[6]
new_image_path = "/content/MDLP_Dashboard_RUB_CNTN+NWRN_new.png"
picture_replace_by_name(slide, "Рисунок Total", new_image_path)
new_image_path = "/content/MDLP_Dashboard_RUB_1_month_new.png"
picture_replace_by_name(slide, "Рисунок month", new_image_path)
slide = prs.slides[7]
new_image_path = "/content/MDLP_Dashboard_RUB_CNTN_new.png"
picture_replace_by_name(slide, "Рисунок CNTN", new_image_path)
new_image_path = "/content/MDLP_Dashboard_RUB_NWRN_new.png"
picture_replace_by_name(slide, "Рисунок NWRN", new_image_path)
slide = prs.slides[8]
new_image_path = "/content/MDLP_Pivot_CNT_N+NWR_N_new.png"
picture_replace(slide, new_image_path)
slide = prs.slides[9]
new_image_path = "/content/MDLP_Pivot_CNT_N_new.png"
picture_replace_by_name(slide, "Рисунок CNTN", new_image_path)
new_image_path = "/content/MDLP_Pivot_NWR_N_new.png"
picture_replace_by_name(slide, "Рисунок NWRN", new_image_path)
slide = prs.slides[10]
new_image_path = "/content/Secondary Sales Report_Distr_MAT_RUB_new.png"
picture_replace_by_name(slide, "Рисунок MAT", new_image_path)
new_image_path = "/content/Secondary Sales Report_Distr_1_month_RUB_new.png"
picture_replace_by_name(slide, "Рисунок Month", new_image_path)
slide = prs.slides[11]
new_image_path = "/content/Prices_Prices_CNT_N+NWR_N_new.png"
picture_replace(slide, new_image_path)
slide = prs.slides[12]
new_image_path = "/content/Stocks Distributors_Stock_CNT_N_days_new.png"
picture_replace_by_name(slide, "Рисунок CNTN", new_image_path)
new_image_path = "/content/Stocks Distributors_Stock_NWR_N_days_new.png"
picture_replace_by_name(slide, "Рисунок NWRN", new_image_path)
slide = prs.slides[13]
new_image_path = "/content/MDLP_Dashboard_onkaspar_new.png"
picture_replace_by_name(slide, "Рисунок Total Onkaspar", new_image_path)
new_image_path = "/content/MDLP_Pivot_onkaspar_new.png"
picture_replace_by_name(slide, "Рисунок Pivot Onkaspar", new_image_path)
slide = prs.slides[17]
new_image_path = "/content/MDLP_Dashboard_lonsurf_new.png"
picture_replace_by_name(slide, "Рисунок Total Lonsurf", new_image_path)
new_image_path = "/content/MDLP_Pivot_lonsurf_new.png"
picture_replace_by_name(slide, "Рисунок Pivot Lonsurf", new_image_path)
slide = prs.slides[19]
new_image_path = "/content/SFE Активность CD_Dashboard_month_new.png"
picture_replace_by_name(slide, "Рисунок Activity", new_image_path)
new_image_path = "/content/SFE Активность CD_Worktime_new.png"
picture_replace_by_name(slide, "Рисунок Worktime", new_image_path)


да
да
Заменяем картинку с именем: Рисунок Total
Заменяем картинку с именем: Рисунок month
Заменяем картинку с именем: Рисунок CNTN
Заменяем картинку с именем: Рисунок NWRN
да
Заменяем картинку с именем: Рисунок CNTN
Заменяем картинку с именем: Рисунок NWRN
Заменяем картинку с именем: Рисунок MAT
Заменяем картинку с именем: Рисунок Month
да
Заменяем картинку с именем: Рисунок CNTN
Заменяем картинку с именем: Рисунок NWRN
Заменяем картинку с именем: Рисунок Total Onkaspar
Заменяем картинку с именем: Рисунок Pivot Onkaspar
Заменяем картинку с именем: Рисунок Total Lonsurf
Заменяем картинку с именем: Рисунок Pivot Lonsurf
Заменяем картинку с именем: Рисунок Activity
Заменяем картинку с именем: Рисунок Worktime


In [ ]:
# Закрываем и сохраняем
prs.save("final_report_1.pptx")
print("Шаблон успешно отредактирован и сохранен!")

Шаблон успешно отредактирован и сохранен!


In [ ]:
slide = prs.slides[17]

# Перебираем все объекты на слайде
for shape in slide.shapes:
    if shape.shape_type == 13:  # Если это картинка

        # Разрываем связь со слайдом №14:
        # Читаем новый файл картинки в бинарном режиме и принудительно записываем в поток
        if "Pivot" in shape.name:
            with open('path_to_pivot.png', 'rb') as f:
                shape.image.blob = f.read()
            print(f"Успешно заменена картинка: {shape.name}")

        elif "Total" in shape.name:
            with open('path_to_total.png', 'rb') as f:
                shape.image.blob = f.read()
            print(f"Успешно заменена картинка: {shape.name}")

# Сохраняем результат под новым именем
prs.save('fixed_presentation.pptx')

In [ ]:
slide_indices = [13, 17]

for index in slide_indices:
    slide = prs.slides[index]
    print(f"\n--- Слайд №{index + 1} (индекс {index}) ---")

    has_pictures = False
    for i, shape in enumerate(slide.shapes):
        # 13 — это тип MSO_SHAPE_TYPE.PICTURE
        if shape.shape_type == 13:
            has_pictures = True
            image = shape.image

            print(f"  [Объект {i}] Картинка: {shape.name}")
            # Хэш SHA-1 уникален для каждого файла картинки в архиве PPTX
            print(f"  └─ Уникальный ID (SHA-1): {image.sha1}")

    if not has_pictures:
        print("  На слайде не найдено картинок.")


--- Слайд №14 (индекс 13) ---
  [Объект 0] Картинка: Рисунок Pivot Onkaspar
  └─ Уникальный ID (SHA-1): 83ffb680c40eb8d7f4319a2d1e8f845b520ad815
  [Объект 6] Картинка: Рисунок Total Onkaspar
  └─ Уникальный ID (SHA-1): eb4bafd34daade55b4c4e1144e269b76c69390c1

--- Слайд №18 (индекс 17) ---
  [Объект 0] Картинка: Рисунок Pivot Lonsurf
  └─ Уникальный ID (SHA-1): 69a95440b379d6fba03b53f16711e941da522af4
  [Объект 6] Картинка: Рисунок Total Lonsurf
  └─ Уникальный ID (SHA-1): 7b7e59d96948dd5c3f95b17f9dc9ef7945a5d47b


In [ ]:
for i, shape in enumerate(prs.slides[13].shapes):
    print(f"Индекс: {i}, Тип: {shape.shape_type}, Текст: {shape.text if shape.has_text_frame else 'Нет текста'}")

Индекс: 0, Тип: PICTURE (13), Текст: Нет текста
Индекс: 1, Тип: PLACEHOLDER (14), Текст: Онкаспар. Продажи MDLP. Май.26. Month. Units.
Индекс: 2, Тип: PLACEHOLDER (14), Текст: Комментарии:
 На YTD план отставание от цели на -10% (факт 114 фл. против цели 127фл.)
Продажи за июнь составили 17 фл., (22 фл с учетом Череповца, который пока не отражается в МДЛП) ниже чем планировали, т.к. снизился спрос в Вологде, Ярославле и Иваново в связи с отсутствием пациентов.
Индекс: 3, Тип: TEXT_BOX (17), Текст: Субъекты территории  
Индекс: 4, Тип: TEXT_BOX (17), Текст: Территория total
Индекс: 5, Тип: TEXT_BOX (17), Текст: По субъектам
Индекс: 6, Тип: PICTURE (13), Текст: Нет текста


In [ ]:
pip install Pillow

In [ ]:
#Этот код предназначен для автоматической нарезки (кроппинга) исходного изображения на отдельные фрагменты по заранее заданным координатам и сохранения каждого фрагмента в виде отдельного файла.
from PIL import Image
import os

# Assuming `img_path1` is the source image for all fragments.
# If you intend to crop from a different image, please update `input_image_path`.
input_image_path = img_path1

# Ensure the input image exists before proceeding
if not os.path.exists(input_image_path):
    print(f"Input image not found: {input_image_path}. Please make sure it exists.")
else:
    # Open the image once to get its dimensions and for cropping
    img = Image.open(input_image_path)
    img_width, img_height = img.size
    print(f"Original image dimensions: {img_width}x{img_height}")

    # Iterate through the `arr_pixels` to crop each fragment
    # `arr_pixels` format: (left, top, width, height, 'name')
    for i, fragment_data in enumerate(arr_pixels):
        left, top, width, height, name = fragment_data

        # Calculate the `right` and `bottom` coordinates required by Pillow's crop method
        right = left + width
        bottom = top + height

        # Ensure the cropping coordinates are within the bounds of the original image
        # This prevents errors if a defined fragment goes beyond the image edges.
        safe_left = max(0, left)
        safe_top = max(0, top)
        safe_right = min(img_width, right)
        safe_bottom = min(img_height, bottom)

        print(f"\nProcessing fragment '{name}':")
        print(f"  Defined region (L, T, W, H): ({left}, {top}, {width}, {height})")
        print(f"  Cropping box (L, T, R, B): ({safe_left}, {safe_top}, {safe_right}, {safe_bottom})")

        # Perform the crop using the safe coordinates
        cropped_img = img.crop((safe_left, safe_top, safe_right, safe_bottom))

        # Define the output path for each cropped image using its descriptive name
        output_image_path = f"/content/cropped_{name}.png"

        # Save the cropped image
        cropped_img.save(output_image_path)
        print(f"  Cropped image saved to: {output_image_path}")

        # Optional: Display the cropped image (uncomment if you want to see them directly)
        # display(cropped_img)

    print("\nAll fragments processed and saved.")

NameError: name 'img_path1' is not defined

In this example:

1.  **`Image.open(input_image_path)`**: Opens the image you want to crop.
2.  **`left`, `top`, `right`, `bottom`**: These define the coordinates of the bounding box for the crop. The coordinates are in pixels, starting from the top-left corner of the image.
    *   `left`: The x-coordinate of the left edge of the cropping box.
    *   `top`: The y-coordinate of the top edge of the cropping box.
    *   `right`: The x-coordinate of the right edge of the cropping box.
    *   `bottom`: The y-coordinate of the bottom edge of the cropping box.
3.  **`img.crop((left, top, right, bottom))`**: Performs the actual cropping based on the defined box.
4.  **`cropped_img.save(output_image_path)`**: Saves the new, cropped image to a specified path.

First, we need to install OpenCV if you don't have it already. Then, we'll import the necessary libraries.

In [ ]:
pip install opencv-python numpy # перейти в папку где лежит "C:\Python313\python.exe"и набрать команду python -m pip install имя_пакета (for Windows)

In [ ]:
#эти данные загружены из файла массив
arr = [


    ['MDLP_Dashboard_lonsurf.png', 'Template_08.png'],
    ['MDLP_Pivot_lonsurf.png', 'Template_09.png']
]

In [ ]:
#эти полные данные загружены из файла массив
arr_all = [
    ['IQDATA MarketData_CNT-N, RUR, month.png', 'Template_01.png'],
    ['IQDATA MarketData_NWR-N, RUR, month.png', 'Template_01.png'],
    ['MDLP_Dashboard_RUB_CNTN+NWRN.png', 'Template_02_1.png'],
    ['MDLP_Dashboard_RUB_1_month.png', 'Template_02_2.png'],
    ['MDLP_Dashboard_RUB_CNTN.png', 'Template_03.png'],
    ['MDLP_Dashboard_RUB_NWRN.png', 'Template_03.png'],
    ['MDLP_Pivot_CNT_N+NWR_N.png', 'Template_04.png'],
    ['MDLP_Pivot_CNT_N.png', 'Template_04.png'],
    ['MDLP_Pivot_NWR_N.png', 'Template_04.png'],
    ['Secondary Sales Report_Distr_MAT_RUB.png', 'Template_05.png'],
    ['Secondary Sales Report_Distr_1_month_RUB.png', 'Template_05.png'],
    ['Prices_Prices_CNT_N+NWR_N.png', 'Template_06.png'],
    ['Stocks Distributors_Stock_CNT_N_days.png', 'Template_07_1.png'],
    ['Stocks Distributors_Stock_NWR_N_days.png', 'Template_07_2.png'],
    ['MDLP_Dashboard_onkaspar.png', 'Template_08.png'],
    ['MDLP_Pivot_onkaspar.png', 'Template_09.png'],
    ['MDLP_Dashboard_lonsurf.png', 'Template_08.png'],
    ['MDLP_Pivot_lonsurf.png', 'Template_09.png'],
    ['SFE Активность CD_Dashboard_month.png', 'Template_10.png'],
    ['SFE Активность CD_Worktime.png', 'Template_11.png']
]

Код вырезает из больших картинок фрагмент по темплейту

In [ ]:
# рабочий код, файлы в формате png
import cv2
import numpy as np
from PIL import Image as PIL_Image # Renamed Image from PIL to PIL_Image
import os

for i in range(len(arr)):
    print(arr[i][0])
    print(arr[i][1])
    # Define the paths for your large image and the fragment image
    large_image_path = f"/content/{arr[i][0]}"
    fragment_image_path = f"/content/{arr[i][1]}"
    file_name_ext = os.path.basename(large_image_path)
    file_name = os.path.splitext(file_name_ext)[0]
    output_found_fragment_path = f"/content/{file_name}_new.png"
    #reader = easyocr.Reader(['ru', 'en'])

    # Check if the images exist
    if not os.path.exists(large_image_path):
        print(f"Error: Large image not found at {large_image_path}")
    elif not os.path.exists(fragment_image_path):
        print(f"Error: Fragment image not found at {fragment_image_path}")
    else:
        # Load the large image and the fragment image using OpenCV, ensuring they are 3-channel BGR
        # cv2.IMREAD_COLOR flag forces a 3-channel image, discarding any alpha channel if present.
        img_large = cv2.imread(large_image_path, cv2.COLOR_BGR2GRAY)
        img_fragment = cv2.imread(fragment_image_path, cv2.COLOR_BGR2GRAY)

        if img_large is None:
            print(f"Error: Could not load large image from {large_image_path}")
        elif img_fragment is None:
            print(f"Error: Could not load fragment image from {fragment_image_path}")
        else:
            # Get the dimensions of the fragment image
            h, w = img_fragment.shape[0], img_fragment.shape[1]

            # Perform template matching
            # cv2.TM_CCOEFF_NORMED is a good method for matching
            result = cv2.matchTemplate(img_large, img_fragment, cv2.TM_CCOEFF_NORMED) #cv2.TM_CCOEFF_NORMED TM_CCORR_NORMED TM_SQDIFF_NORMED

            # Find the best match location
            min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)

            # max_loc gives the top-left corner (x, y) of the best match
            top_left = max_loc
            bottom_right = (top_left[0] + w, top_left[1] + h)

            print(f"Fragment found with confidence: {max_val:.2f}")
            print(f"Top-left corner of match: {top_left}")
            print(f"Bottom-right corner of match: {bottom_right}")

            # Crop the found fragment from the original large image using PIL for consistency
            # Re-open with PIL for cropping (Pillow handles original image channels correctly)
            img_large_pil = PIL_Image.open(large_image_path) # Use PIL_Image.open()

            # Define the bounding box for cropping (left, upper, right, lower)
            # Note: top_left[0] is x-coordinate (left), top_left[1] is y-coordinate (upper)
            left = top_left[0]
            upper = top_left[1]
            right = bottom_right[0]
            lower = bottom_right[1]

            cropped_found_fragment_pil = img_large_pil.crop((left, upper, right, lower))

            # Save the cropped fragment
            cropped_found_fragment_pil.save(output_found_fragment_path)
            print(f"Found and cropped fragment saved to: {output_found_fragment_path}")

        # Optional: Draw a rectangle around the found object on the large image
        # This requires converting back to OpenCV format if you want to save it with the rectangle
        # img_display = cv2.cvtColor(img_large, cv2.COLOR_BGR2RGB) # Convert to RGB if originally BGR
        # cv2.rectangle(img_display, top_left, bottom_right, (255, 0, 0), 2) # Red rectangle, 2px thickness
        # cv2.imwrite("/content/large_image_with_match.png", img_display)
        # print("Large image with highlighted match saved to: /content/large_image_with_match.png"

MDLP_Dashboard_lonsurf.png
Template_08.png
Fragment found with confidence: 0.58
Top-left corner of match: (255, 218)
Bottom-right corner of match: (1266, 885)
Found and cropped fragment saved to: /content/MDLP_Dashboard_lonsurf_new.png
MDLP_Pivot_lonsurf.png
Template_09.png
Fragment found with confidence: 0.53
Top-left corner of match: (242, 240)
Bottom-right corner of match: (1108, 560)
Found and cropped fragment saved to: /content/MDLP_Pivot_lonsurf_new.png


### Explanation:

1.  **`pip install opencv-python numpy`**: Installs the necessary libraries.
2.  **Load Images**: Both the `large_image_path` (your original `img_path1`) and the `fragment_image_path` (the smaller image you want to find) are loaded.
3.  **`cv2.matchTemplate()`**: This is the core function. It slides the `img_fragment` over the `img_large` and calculates a matching score. `cv2.TM_CCOEFF_NORMED` is a good method for general matching, providing a normalized score.
4.  **`cv2.minMaxLoc()`**: This function finds the minimum and maximum values and their locations in the `result` array from `matchTemplate`. `max_loc` will give you the top-left (x, y) coordinates of the best match.
5.  **Calculate Bounding Box**: From `max_loc` (top-left) and the dimensions of the fragment (`w`, `h`), we calculate the `bottom_right` coordinates.
6.  **Crop with Pillow**: We re-open the large image using Pillow (`PIL.Image.open`) and use its `crop()` method with the calculated `left`, `upper`, `right`, `lower` coordinates to extract the found fragment.
7.  **Save**: The `cropped_found_fragment_pil` is then saved.

You can display the cropped fragment and the original large image using the following code:

In [ ]:
from IPython.display import Image, display

print(f"Displaying the found and cropped fragment from: {output_found_fragment_path}")
display(Image(filename=output_found_fragment_path))

print(f"\nDisplaying the original large image from: {large_image_path}")
display(Image(filename=large_image_path))

NameError: name 'output_found_fragment_path' is not defined

In [ ]:
 cv2.minMaxLoc(result)


(-0.0776064321398735, 0.7598161101341248, (386, 3), (251, 164))

In [ ]:
folder_path = "/content/"  # Укажите путь к вашей папке

# Получаем список всего содержимого папки
all_items = os.listdir(folder_path)

# Фильтруем, чтобы оставить только файлы (исключаем подпапки)
files_only = [f for f in all_items if os.path.isfile(os.path.join(folder_path, f))]


In [ ]:
print(files_only)


['Template_10.png', 'Stocks Distributors_Stock_NWR_N.png', 'Secondary Sales Report_Distr_1_month_RUB.png', 'MDLP_Pivot_CNT_N+NWR_N.png', 'MDLP_Dashboard_RUB_CNTN_new.png', 'Prices_Prices_CNT_N+NWR_N.png', 'Отчет РТМ.pptx', 'MDLP_Dashboard_onkaspar_new.png', 'MDLP_Dashboard_RUB_CNTN+NWRN.png', 'MDLP_Pivot_CNT_N+NWR_N_new.png', 'Template_01.png', 'MDLP_Pivot_NWR_N_new.png', 'Template_04.png', 'MDLP_Pivot_CNT_N.png', 'Template_05.png', 'Stocks Distributors_Stock_NWR_N_days_new.png', 'Secondary Sales Report_Distr_MAT_RUB.png', 'MDLP_Pivot_NWR_N.png', 'MDLP_Dashboard_RUB_CNTN+NWRN_new.png', 'MDLP_Pivot_CNT_N_new.png', 'IQDATA MarketData_NWR-N, RUR, month.png', 'Stocks Distributors_Stock_CNT_N.png', 'MDLP_Dashboard_onkaspar.png', 'Template_02.png', 'SFE Активность CD_Worktime_new.png', 'IQDATA MarketData_NWR-N, RUR, month_new.png', 'Template_03.png', 'SFE Активность CD_Dashboard_month.png', 'Template_06.png', 'MDLP_Dashboard_RUB_NWRN.png', 'Template_11.png', 'MDLP_Dashboard_RUB_NWRN_new.png'

In [ ]:
arr = [
    ['IQDATA MarketData_CNT-N, RUR, month.png', 'Template_1.png'],
    ['IQDATA MarketData_NWR-N, RUR, month.png', 'Template_1.png'],
    ['MDLP_Dashboard_RUB_CNTN+NWRN.png', 'Template_2.png'],
    ['MDLP_Dashboard_RUB_CNTN.png', 'Template_3.png'],
    ['MDLP_Dashboard_RUB_NWRN.png', 'Template_3.png'],
    ['MDLP_Pivot_CNT_N+NWR_N.png', 'Template_4.png'],
    ['MDLP_Pivot_CNT_N.png', 'Template_5.png'],
    ['MDLP_Pivot_NWR_N.png', 'Template_5.png'],
    ['Secondary Sales Report_Distr_MAT_RUB.png', 'Template_6.png'],
    ['Secondary Sales Report_Distr_1_month_RUB.png', 'Template_6.png'],
    ['Prices_Prices_CNT_N+NWR_N.png', 'Template_7.png'],
    ['Stocks Distributors_Stock_CNT_N_days.png', 'Template_8.png'],
    ['Stocks Distributors_Stock_NWR_N_days.png', 'Template_8.png'],
    ['MDLP_Dashboard_onkaspar.png', 'Template_9.png'],
    ['SFE Активность CD_Dashboard_month.png', 'Template_11.png'],
    ['SFE Активность CD_Worktime.png', 'Template_12.png']
]

In [ ]:
arr[4][1]

'Template_3.png'

In [ ]:
for i in range(len(arr)):
    print(arr[i][0])
    print(arr[i][1])

IQDATA MarketData_CNT-N, RUR, month.png
Template_1.png
IQDATA MarketData_NWR-N, RUR, month.png
Template_1.png
MDLP_Dashboard_RUB_CNTN+NWRN.png
Template_2.png
MDLP_Dashboard_RUB_CNTN.png
Template_3.png
MDLP_Dashboard_RUB_NWRN.png
Template_3.png
MDLP_Pivot_CNT_N+NWR_N.png
Template_4.png
MDLP_Pivot_CNT_N.png
Template_5.png
MDLP_Pivot_NWR_N.png
Template_5.png
Secondary Sales Report_Distr_MAT_RUB.png
Template_6.png
Secondary Sales Report_Distr_1_month_RUB.png
Template_6.png
Prices_Prices_CNT_N+NWR_N.png
Template_7.png
Stocks Distributors_Stock_CNT_N_days.png
Template_8.png
Stocks Distributors_Stock_NWR_N_days.png
Template_8.png
MDLP_Dashboard_onkaspar.png
Template_9.png
SFE Активность CD_Dashboard_month.png
Template_11.png
SFE Активность CD_Worktime.png
Template_12.png


Эта часть тестирует отчеты BI

In [ ]:
# словарь с названиями отчетов и закладками
bookmarks = {
    "MDLP": ["Dashboard_RUB_CNTN", "Dashboard_RUB_NWRN", "Dashboard_RUB_CNTN+NWRN", "DB_RUB_1_month", "Pivot_CNT_N", "Pivot_NWR_N", "Pivot_CNT_N+NWR_N", "Dashboard_onkaspar", "Pivot_onkaspar"],
    "SFE Активность CD": ["Dashboard_month", "Worktime"],
    "IQDATA MarketData": ["CNT-N, RUR, month", "NWR-N, RUR, month"],
    "Secondary Sales Report": ["Distr_1_month_RUB", "Distr_MAT_RUB"],
    "Stocks Distributors": ["Stock_CNT_N", "Stock_NWR_N", "Stock_CNT_N_days", "Stock_NWR_N_days"],
    "Prices": ["Prices_CNT_N+NWR_N"]
}
for report_name, tabs in bookmarks.items():
    print(f"📋 Отчет: {report_name}")
    for tab in tabs:
        print(f"   🔹 Закладка: {tab}")
    print("-" * 20)


📋 Отчет: MDLP
   🔹 Закладка: Dashboard_RUB_CNTN
   🔹 Закладка: Dashboard_RUB_NWRN
   🔹 Закладка: Dashboard_RUB_CNTN+NWRN
   🔹 Закладка: DB_RUB_1_month
   🔹 Закладка: Pivot_CNT_N
   🔹 Закладка: Pivot_NWR_N
   🔹 Закладка: Pivot_CNT_N+NWR_N
   🔹 Закладка: Dashboard_onkaspar
   🔹 Закладка: Pivot_onkaspar
--------------------
📋 Отчет: SFE Активность CD
   🔹 Закладка: Dashboard_month
   🔹 Закладка: Worktime
--------------------
📋 Отчет: IQDATA MarketData
   🔹 Закладка: CNT-N, RUR, month
   🔹 Закладка: NWR-N, RUR, month
--------------------
📋 Отчет: Secondary Sales Report
   🔹 Закладка: Distr_1_month_RUB
   🔹 Закладка: Distr_MAT_RUB
--------------------
📋 Отчет: Stocks Distributors
   🔹 Закладка: Stock_CNT_N
   🔹 Закладка: Stock_NWR_N
   🔹 Закладка: Stock_CNT_N_days
   🔹 Закладка: Stock_NWR_N_days
--------------------
📋 Отчет: Prices
   🔹 Закладка: Prices_CNT_N+NWR_N
--------------------


In [ ]:
import json
def get_value(data, *args):
    # Метод .get() защитит от ошибок, если ключа нет
    for arg in args:
        data = data[arg]
    return data

with open(r"/content/links.json", "r", encoding="utf-8") as file:
    reports = json.load(file)



In [ ]:
for report_name, report_data in reports.items():
    print(f"Отчет: {report_name}")
    print(get_value(reports, report_name, "url"))

    if "bookmarks" in report_data and isinstance(report_data["bookmarks"], dict):
        for bookmark_name in report_data["bookmarks"].keys():
            print(f"Закладка: {bookmark_name}")
            print(get_value(reports, report_name, "bookmarks", bookmark_name))
    else:
        print("   Нет закладок для этого отчета.")

    print("-" * 20)

Отчет: MDLP
https://app.powerbi.com/groups/de1ee668-d338-4a6a-af41-7a8561a36550/reports/a4e2a45c-d340-453e-a637-728636f0ac76/ReportSection6506c7a0a7d30cb0320a?language:en-US&experience:power-bi
Закладка: Dashboard_RUB_CNTN
Dashboard_RUB_CNTN
Закладка: Dashboard_RUB_NWRN
Dashboard_RUB_NWRN
Закладка: Dashboard_RUB_CNTN+NWRN
Dashboard_RUB_CNTN+NWRN
Закладка: Pivot_CNT_N
Pivot_CNT_N
Закладка: Pivot_NWR_N
Pivot_NWR_N
Закладка: Pivot_CNT_N+NWR_N
Pivot_CNT_N+NWR_N
Закладка: Dashboard_onkaspar
Dashboard_onkaspar
Закладка: Dashboard_RUB_1_month
Dashboard_RUB_1_month
Закладка: Pivot_onkaspar
Pivot_onkaspar
Закладка: Dashboard_lonsurf
Dashboard_lonsurf
Закладка: Pivot_lonsurf
Pivot_lonsurf
--------------------
Отчет: SFE Активность CD
https://app.powerbi.com/groups/me/reports/682a28ae-95ca-4bb7-86dc-6c74fc5d6522?ctid:cc0a4ff6-9454-4e4b-881b-85f448dee2e3&experience:power-bi
Закладка: Dashboard_month
Dashboard_month
Закладка: Worktime
Worktime
--------------------
Отчет: IQDATA MarketData
https://a

In [ ]:
pip install playwright

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 35.1 MB/s eta 0:00:00


In [ ]:
from datetime import datetime
import json
import os
import re
from playwright.sync_api import sync_playwright
from playwright.sync_api import expect, TimeoutError as PlaywrightTimeoutError


In [ ]:
playwright codegen

SyntaxError: invalid syntax (803514875.py, line 1)

In [ ]:
with sync_playwright() as p:
    # Указываем папку для сохранения сессии и отключаем флаг автоматизации
    browser = p.chromium.launch_persistent_context(channel="msedge", slow_mo=3000, user_data_dir="./user_data", headless=False,args=["--disable-blink-features=AutomationControlled"]) # Маскировка от ботов
    page = browser.new_page()
    page.set_viewport_size({"width": 1920, "height": 1080}) # Размер важен, все темплейты нарезаны из такого разрешения
    target_folder = create_screenshot_folder()

    # Открываем файл json в режиме чтения с явным указанием кодировки utf-8
    with open(r"C:\Users\BVP_RU1\Создать_отчет\links.json", "r", encoding="utf-8") as file:
        reports = json.load(file)



    ###print(get_value(target_urls,"MDLP","pages","Pivot Table")) работает
    ###print(reports['MDLP']['months']["10-2025"]) работает

    for report_name, tabs in reports_dict.items():
        print(f"Отчет: {report_name}")
        page_goto(get_value(reports,report_name,"url"))
        for tab in tabs:
            print(f"Закладка: {tab}")
            select_bookmark(get_value(reports,report_name,"bookmarks",tab))
            file_name = f"{get_file_name(reports,report_name,"bookmarks",tab)}.png"
            print(file_name)
            take_screenshot(target_folder, file_name)

        print("-" * 20)

    # Начало теста на закладки

    #page_goto(get_value(reports,"MDLP","url"))
    #options_month = ("Apr.2026","Sep.2026", )
    #select_multiple_checkboxes_scroll_1("Month-Week-Day", options_month)




    browser.close()
    # Конец теста


Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.